In [18]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer

from scipy.stats import ks_2samp

import shap

In [19]:
df_lc = pd.read_csv('../../data/processed/featured_lending-club-car-loan.csv')
df_lt = pd.read_csv('../../data/processed/featured_lt-vehicle-loan_train.csv')

In [20]:
print("LendingClub Shape:", df_lc.shape)
print("L&T Shape:", df_lt.shape)

print("\nLendingClub Columns:")
print(df_lc.columns.tolist())

print("\nL&T Columns:")
print(df_lt.columns.tolist())

LendingClub Shape: (1373915, 24)
L&T Shape: (233154, 14)

LendingClub Columns:
['Unnamed: 0', 'int_rate', 'term', 'emp_length', 'revol_util', 'total_acc', 'emp_length_missing', 'credit_score_norm', 'inquiry_intensity', 'log_delinquency', 'log_loan_amount', 'log_installment', 'log_income', 'log_credit_history_length', 'log_inquiries', 'target', 'log_dti', 'log_loan_to_income', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'credit_history_length']

L&T Columns:
['Unnamed: 0', 'credit_score', 'target', 'ltv_norm', 'loan_to_asset', 'credit_score_norm', 'log_installment', 'log_inquiries', 'log_active_accounts', 'log_loan_amount', 'log_delinquency', 'log_credit_history_length', 'Employment.Type_Self_employed', 'Employment.Type_Unknown']


In [21]:
aligned_features = [
     "log_loan_amount",
    "log_installment",
    # "log_credit_history_length",
    # "log_inquiries",
    "log_delinquency"
]

In [24]:
lc_aligned = df_lc[aligned_features].copy()
lt_aligned = df_lt[aligned_features].copy()

# Add Target
lc_aligned["target"] = df_lc["target"]
lt_aligned["target"] = df_lt["target"]

# Source Label
lc_aligned["source"] = "LC"
lt_aligned["source"] = "LT"

In [25]:
# Combine Datasets
df_aligned = pd.concat([lc_aligned, lt_aligned], axis=0)

In [30]:
qt = QuantileTransformer(
    output_distribution="normal",
    random_state=42
)

# Split
lc_part = df_aligned[df_aligned["source"] == "LC"].copy()
lt_part = df_aligned[df_aligned["source"] == "LT"].copy()

# Fit on LC
qt.fit(lc_part[aligned_features])

# Transform both
lc_part[aligned_features] = qt.transform(lc_part[aligned_features])
lt_part[aligned_features] = qt.transform(lt_part[aligned_features])

# Recombine
df_aligned = pd.concat([lc_part, lt_part])

In [31]:
df_aligned.groupby("source")[aligned_features].mean()
df_aligned.groupby("source")[aligned_features].std()

,log_loan_amount,log_installment,log_delinquency
source,,,
LC,1.077974,1.004984,2.607179
LT,0.412086,2.069800,1.720085


In [32]:
X = df_aligned[aligned_features]
y = (df_aligned["source"] == "LC").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict_proba(X_test)[:, 1]
print("Source AUC:", roc_auc_score(y_test, y_pred))

Source AUC: 0.9989710273531845


Full distribution alignment is NOT possible